# REWIND batch pipeline: many videos in, results on disk

Runs the ingest pipeline on every video in `selection.csv` (about 1 h of MEVA + 1 h of UCA by default) and saves
everything to `outputs/batch/`. It shows **no pictures**: look at the results in **`explore.ipynb`**.

| Stage | What it does | Code | Needs |
|---|---|---|---|
| 1. Motion gating | Score 5 frames per second for motion (frame differencing, MOG2), turn the scores into **segments** of activity | `motion_gating.py` | CPU |
| 2. Keyframes | Inside each segment: first frame, peak, every visual change, plus an idle "heartbeat" | `keyframes.py` | CPU |
| GT check | Did the segments and keyframes catch the labelled moments? (MEVA activities, UCA sentences) | `datasets.py`, `motion_gating.py` | CPU |
| 3. Events | Inside segments: YOLO-World + ByteTrack, SigLIP and X-CLIP tags | `gpu_stages.py` | GPU |
| 4. Hand-off | Pick frames for the captioner / index, write `index_records.jsonl` + SigLIP embeddings | `gpu_stages.py` | GPU |
| 5. Captions | Qwen-VL describes each selected frame and each clip (segment cut into ≤ 60 s) as JSON → `captions/captions.csv` | `captions.py` | GPU (~16 GB) |

It runs in three phases: **stages 1-2 for every video first**, then stages 3-4, then stage 5 (captions). So a GPU
problem never costs you the CPU results. Captions use stage 4's frames if they exist, otherwise stage 2's keyframes. A video that fails is logged and skipped. Re-running skips finished videos (`FORCE=False`).

**To run:** check the config cell, then *Run All*. To pick other videos, edit `selection.csv` (or delete it to
pick again). The same thing runs from a terminal: `python batch.py` (inside `tmux`, it survives a closed laptop).

In [ ]:
import os
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

if Path("/mnt/5T_storage").exists():  # team VM: Hugging Face models go to the 5T disk, not the nearly full root disk
    os.environ.setdefault("HF_HOME", "/mnt/5T_storage/hf_cache")

import batch
import datasets as ds
import motion_gating as mg

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

## Config

In [ ]:
DATA_ROOT = Path(os.environ.get("REWIND_DATA_ROOT", "/mnt/5T_storage/clipsExtraction/datasets"))
OUT_ROOT = Path(os.environ.get("REWIND_OUT_ROOT", "outputs/batch"))   # one folder per video: <dataset>/<video>/
SELECTION_CSV = Path(os.environ.get("REWIND_SELECTION", "selection.csv"))  # reused if it exists; edit it to change videos
REMAKE_SELECTION = False   # True = pick the videos again (overwrites selection.csv)
MEVA_MINUTES = 60          # MEVA: whole 5-min time slots of all cameras until this many minutes (annotated slots first)
UCA_MINUTES = 60           # UCA: spread over categories (test split first) until this many minutes
UCA_VIDEO_DIR = "dev_videos"
MEVA_CAMERAS = None        # e.g. ["G328", "G336"]; None = every camera

STAGES = ["cpu", "gpu", "vlm"]   # ["cpu"] = only stages 1-2 (fast, no GPU); ["vlm"] = only captions
FORCE = False              # True = redo videos that already have results
MOCK_MODELS = os.environ.get("REWIND_MOCK", "") == "1"   # fake GPU models: plumbing check only, numbers meaningless

CFG = {
    "out_root": OUT_ROOT,
    "gate": "mog2",        # gate that feeds keyframes and stages 3-4: "mog2" | "diff" | "fused" | "deep"
    "deep_gate": False,    # DINOv2 gate = a GPU pass over every whole video; docs/pipeline.md found it not worth it
    "scan": dict(          # stage 1, slow pass (changing these re-decodes the videos)
        sample_fps=5.0, width=320, blur_ksize=5, diff_threshold=15,
        mog2_history=500, mog2_var_threshold=16.0, mog2_learning_rate=-1.0,
        mog2_illum_ncc=0.7, mog2_color=True, morph_kernel=3, ignore_polygons=[], hash_grid=4, hist_bins=32,
    ),
    "segments": dict(      # stage 1, cheap
        gates=["diff", "mog2"], threshold_mode="adaptive", k=3.0, q_noise=0.5,
        floor={"diff": 0.001, "mog2": 0.001, "deep": 0.004}, fixed_threshold={"mog2": 0.004, "diff": 0.006},
        stop_ratio=0.5, smooth_s=1.0, merge_gap_s=3.0, min_duration_s=1.0, pad_s=2.0,
    ),
    "keyframes": dict(     # stage 2, cheap
        metric="phash", change_threshold=6.0, dedup_threshold=2.5, min_gap_s=3.0, max_gap_s=30.0, heartbeat_s=300.0,
    ),
    "v2": {                # stages 3-4 (all keys: gpu_stages.DEFAULT_CONFIG)
        "sampling": {"primary": "kcenter", "export_jpgs": ["kcenter"]},
        "runtime": {"device": "auto", "mock_models": MOCK_MODELS},
    },
    "vlm": {               # stage 5 captions (all keys: captions.CAPTION_CONFIG)
        "model": "Qwen/Qwen2.5-VL-7B-Instruct",   # HF name or local folder of the weights on the VM (Qwen3-VL works too)
        "levels": ["frames", "clips"],   # one caption per selected frame / per clip (segment cut into <= clip_max_s)
        "source": "auto",                # "index" = stage 4 frames, "keyframes" = stage 2, "auto" = index if present
        "clip_max_s": 60.0, "clip_max_frames": 6, "batch_size": 4,
        "mock": MOCK_MODELS,
    },
    "review_n": 60,        # frames in outputs/batch/caption_review.csv, for the hand check of caption quality
}

## 1. What is on disk
All videos of both datasets, before picking. Durations come from the file names (MEVA) and the annotations (UCA).

In [ ]:
meva_inv = ds.meva_inventory(DATA_ROOT)
uca_inv = ds.uca_inventory(DATA_ROOT, UCA_VIDEO_DIR)
inv = pd.concat([meva_inv, uca_inv], ignore_index=True)
display(ds.describe(inv))
if len(meva_inv):
    print("MEVA cameras:", meva_inv.camera_id.value_counts().to_dict(),
          "| clips shorter than 4.5 min:", int((meva_inv.duration_s < 270).sum()),
          "| annotation matches:", meva_inv.gt_match.value_counts().to_dict())
if len(uca_inv):
    print("UCA splits:", uca_inv.split.replace("", "(no annotation)").value_counts().to_dict())
    print("UCA categories:", uca_inv.category.value_counts().to_dict())

## 2. Which videos
`selection.csv` lists them (path, camera, wall-clock start, ground-truth file). Edit it by hand to change the set.

In [ ]:
sel = ds.load_or_make_selection(SELECTION_CSV, DATA_ROOT, remake=REMAKE_SELECTION, meva_min=MEVA_MINUTES,
                                uca_min=UCA_MINUTES, uca_video_dir=UCA_VIDEO_DIR, meva_cameras=MEVA_CAMERAS)
missing = [p for p in sel.path if not Path(p).exists()]
if missing:
    print(f"WARNING: {len(missing)} files in {SELECTION_CSV} don't exist, e.g. {missing[0]}")
display(sel.assign(minutes=sel.duration_s.astype(float) / 60).groupby("dataset").agg(
    videos=("video_id", "size"), minutes=("minutes", "sum"),
    with_ground_truth=("gt_match", lambda s: int((s != "none").sum()))).round(1))
display(sel[["dataset", "video_id", "camera_id", "start_wallclock", "duration_s", "category", "split", "gt_match", "n_gt"]])

## 3. Run
One line per video. Results: `outputs/batch/<dataset>/<video>/` (same files as before: `segments.csv`, `keyframes/`,
`motion_scores.csv`, `gt_eval_*.csv`, `events/`...). Errors are saved next to them in `error_cpu.json` / `error_gpu.json` / `error_vlm.json`. Captions: `captions/`.

In [ ]:
t0 = time.perf_counter()
summary = batch.run_batch(sel, CFG, stages=STAGES, force=FORCE, log=lambda m: print(m, flush=True))
print(f"\ntotal {(time.perf_counter() - t0) / 60:.1f} min")

## 4. Summary
Per dataset. Details, pictures and failure examples: **`explore.ipynb`**.

In [ ]:
s = summary.copy()
ok = s[s.get("status_cpu", pd.Series(dtype=str)).astype(str) == "ok"] if "status_cpu" in s else s.iloc[0:0]
if len(ok):
    g = ok.groupby("dataset")
    tab = pd.DataFrame({
        "videos": g.size(),
        "hours": g.duration_s.sum() / 3600,
        "active %": g.apply(lambda d: (d.active_pct * d.duration_s).sum() / d.duration_s.sum()),
        "keyframes": g.keyframes.sum(),
        "reduction (x fewer than 1 fps)": g.apply(lambda d: d.naive_frames.sum() / max(1, d.keyframes.sum())),
        "CPU min": g.cpu_s.sum() / 60,
    })
    if "gt_kf_hit" in ok:
        w = ok.dropna(subset=["gt_kf_hit"])
        if len(w):
            gw = w.groupby("dataset")
            tab["GT items"] = gw.gt_items.sum()
            tab["GT items with a keyframe %"] = gw.apply(lambda d: 100 * (d.gt_kf_hit * d.gt_items).sum() / d.gt_items.sum())
            tab["same, uniform sampling %"] = gw.apply(lambda d: 100 * (d.gt_kf_hit_uniform * d.gt_items).sum() / d.gt_items.sum())
    if "gpu_s" in ok:
        tab["GPU min"] = g.gpu_s.sum() / 60
    if "vlm_s" in ok:
        tab["captions (frames + clips)"] = g.vlm_frames.sum() + g.vlm_clips.sum()
        tab["captions unparsed"] = g.vlm_parse_failed.sum()
        tab["VLM min"] = g.vlm_s.sum() / 60
    display(tab.round(2))
errs = s[[c for c in s.columns if c.startswith("status_")]].astype(str).apply(lambda r: r.str.startswith("error").any(), axis=1)
if errs.any():
    display(Markdown(f"**{int(errs.sum())} video(s) with errors:**"))
    display(s.loc[errs, ["dataset", "video_id"] + [c for c in s.columns if c.startswith("status_")]])
print(f"summary table: {OUT_ROOT / 'batch_summary.csv'}")